# GOLD ATP PLAYERS

## Imports

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [9]:
try:
    spark = SparkSession.builder.appName("dim_players").getOrCreate()
except Exception as e:
    print(e)

In [10]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [11]:
# tb_players = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "silver.tb_atp_players")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )

tb_players = spark.read.csv("../../../data/silver/tb_atp_players.csv", sep=',', header=True)

## Players

In [12]:
country_mapping = {
    "ALG": "dz", "ARG": "ar", "ARM": "am", "AUS": "au", "AUT": "at",
    "AZE": "az", "BAH": "bs", "BAR": "bb", "BEL": "be", "BER": "bm",
    "BIH": "ba", "BLR": "by", "BOL": "bo", "BRA": "br", "BUL": "bg",
    "BUR": "bf", "CAN": "ca", "CAR": "cw", "CHI": "cl", "CHN": "cn",
    "CIV": "ci", "COL": "co", "CRC": "cr", "CRO": "hr", "CUB": "cu",
    "CUW": "cw", "CYP": "cy", "CZE": "cz", "DEN": "dk", "DOM": "do",
    "ECU": "ec", "EGY": "eg", "ESA": "sv", "ESP": "es", "EST": "ee",
    "FIN": "fi", "FRA": "fr", "FRG": "de", "GBR": "gb", "GDR": "de",
    "GEO": "ge", "GER": "de", "GRE": "gr", "HAI": "ht", "HKG": "hk",
    "HUN": "hu", "INA": "id", "IND": "in", "IRI": "ir", "IRL": "ie",
    "ISR": "il", "ITA": "it", "JAM": "jm", "JOR": "jo", "JPN": "jp",
    "KAZ": "kz", "KEN": "ke", "KOR": "kr", "KUW": "kw", "LAT": "lv",
    "LBN": "lb", "LTU": "lt", "LUX": "lu", "MAR": "ma", "MAS": "my",
    "MDA": "md", "MEX": "mx", "MKD": "mk", "MON": "mc", "NED": "nl",
    "NGR": "ng", "NIG": "ne", "NOR": "no", "NZL": "nz", "PAK": "pk",
    "PAN": "pa", "PAR": "py", "PER": "pe", "PHI": "ph", "POL": "pl",
    "POR": "pt", "PRT": "pt", "PUR": "pr", "QAT": "qa", "RHO": "zw",
    "ROU": "ro", "RSA": "za", "RUS": "ru", "SEN": "sn", "SLO": "si",
    "SRB": "rs", "SRI": "lk", "SUD": "sd", "SUI": "ch", "SUR": "sr",
    "SVK": "sk", "SVN": "si", "SWE": "se", "TCH": "cz", "THA": "th",
    "TPE": "tw", "TUN": "tn", "TUR": "tr", "UAE": "ae", "UKR": "ua",
    "UNK": "un", "URS": "ru", "URU": "uy", "USA": "us", "UZB": "uz",
    "VEN": "ve", "VIE": "vn", "YUG": "rs", "ZIM": "zw"
}

In [13]:
from itertools import chain

mapping_expr = f.create_map([f.lit(x) for x in chain(*country_mapping.items())])

df = (
    tb_players 
    .withColumn(
        "PLAYER_HAND",
        f.when(f.col("PLAYER_HAND") == 'L', "Left-Handed")
        .when(f.col("PLAYER_HAND") == 'R', "Right-Handed")
        .when(f.col("PLAYER_HAND") == 'A', "Ambidextrous")
        .when(f.col("PLAYER_HAND") == 'U', "Unknown")
        .otherwise("Unknown")
    )
    .withColumn(
        "PLAYER_COUNTRY_ISO2",
        mapping_expr[f.upper(f.trim(f.col("PLAYER_COUNTRY")))]
    )
    .groupBy("PLAYER_NAME")
    .agg(
        f.last("PLAYER_ID").alias("PLAYER_ID"),
        f.first("PLAYER_ID").alias("PLAYER_ID_OLD"),
        f.last("PLAYER_HAND").alias("PLAYER_HAND"),
        f.last("PLAYER_HEIGHT").alias("PLAYER_HEIGHT"),
        f.last("PLAYER_COUNTRY").alias("PLAYER_COUNTRY"),
        f.last("PLAYER_COUNTRY_ISO2").alias("PLAYER_COUNTRY_ISO2"),
        f.last("PLAYER_BIRTH_DATE").alias("PLAYER_BIRTH_DATE"),
        f.first("DATE_INGESTION").alias("DATE_INGESTION")
    )
    .distinct()
    .withColumn("SK_PLAYER", f.monotonically_increasing_id() + 1)
)

## Save dataframe

### Local

In [15]:
df.toPandas().to_csv(
    r"../../../data/gold/dimension/dim_players.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [16]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)